# Train ASL Alphabet Classifier — NIMBUS Fingerspelling

This notebook trains a lightweight **MLP** on MediaPipe hand landmarks to classify
the **26 ASL alphabet letters (A-Z)**, then exports an ONNX model for browser inference.

**Designed for Kaggle** — uses free GPU and the ASL Alphabet dataset (87K images).

### Kaggle Setup
1. Add dataset: **"grassknoted/asl-alphabet"** (87,000 images, A-Z)
2. Enable **GPU accelerator** (Settings → Accelerator → GPU T4 ×2)

### Pipeline
1. Load ASL Alphabet images (A-Z, 3000 per class)
2. Extract 21-point MediaPipe hand landmarks per image
3. Normalize landmarks (wrist-centered, scale-invariant)
4. Train a 3-layer MLP: 42 → 256 → 128 → 26
5. Export to ONNX (opset 14)
6. Download model + labels for `web/public/`

### Key Difference from TGCN Word Model
- **Single frame** — no 50-frame temporal buffer needed
- **Hand only** — 21 landmarks × 2 coords = 42 features (not 55 × 100)
- **~50KB ONNX** vs hundreds of KB for TGCN
- Browser pipeline simplifies: `MediaPipe Hand → normalize → ONNX → letter`

In [ ]:
# ============================================================
# Cell 1: Install Dependencies
# ============================================================
!pip install -q mediapipe==0.10.14 onnx onnxruntime torch torchvision
!pip install -q onnxscript || echo "onnxscript install failed (non-fatal)"

In [ ]:
# ============================================================
# Cell 2: Imports
# ============================================================
import os
import gc
import cv2
import json
import math
import numpy as np
from pathlib import Path
from tqdm import tqdm
import mediapipe as mp
from collections import Counter

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import Dataset, DataLoader

print(f"PyTorch: {torch.__version__}")
print(f"CUDA available: {torch.cuda.is_available()}")
DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Constants & Target Classes

26 ASL alphabet letters (A-Z). Static hand poses only — no temporal component.
Each sample is a single image → 21 MediaPipe hand landmarks → 42 features.

In [ ]:
# ============================================================
# Cell 3: Constants
# ============================================================
LETTERS = [chr(i) for i in range(65, 91)]  # A-Z
NUM_CLASSES = len(LETTERS)
NUM_HAND_LANDMARKS = 21
NUM_FEATURES = NUM_HAND_LANDMARKS * 2  # 42 (x, y per landmark)

letter_to_idx = {l: i for i, l in enumerate(LETTERS)}
idx_to_letter = {i: l for l, i in letter_to_idx.items()}

print(f"Classes: {NUM_CLASSES} — {LETTERS}")
print(f"Features per sample: {NUM_FEATURES}")

## 2. Load ASL Alphabet Dataset

Uses the [grassknoted/asl-alphabet](https://www.kaggle.com/datasets/grassknoted/asl-alphabet)
dataset: 87,000 images (200×200 RGB), 3,000 per letter.

We only use the A-Z folders (skip "del", "nothing", "space").

In [ ]:
# ============================================================
# Cell 4: Locate Dataset
# ============================================================
DATASET_CANDIDATES = [
    "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train/asl_alphabet_train",
    "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_train",
    "/kaggle/input/asl-alphabet/asl_alphabet_train/asl_alphabet_train",
    "/kaggle/input/asl-alphabet/asl_alphabet_train",
]

TEST_CANDIDATES = [
    "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_test/asl_alphabet_test",
    "/kaggle/input/datasets/grassknoted/asl-alphabet/asl_alphabet_test",
    "/kaggle/input/asl-alphabet/asl_alphabet_test/asl_alphabet_test",
    "/kaggle/input/asl-alphabet/asl_alphabet_test",
]

DATASET_DIR = None
for candidate in DATASET_CANDIDATES:
    if os.path.isdir(candidate):
        if os.path.isdir(os.path.join(candidate, "A")):
            DATASET_DIR = candidate
            break

if DATASET_DIR is None:
    raise FileNotFoundError(
        "ASL Alphabet dataset not found. Add 'grassknoted/asl-alphabet' as a Kaggle dataset input."
    )

KAGGLE_TEST_DIR = None
for candidate in TEST_CANDIDATES:
    if os.path.isdir(candidate):
        KAGGLE_TEST_DIR = candidate
        break

# List available class folders
all_folders = sorted([
    d for d in os.listdir(DATASET_DIR)
    if os.path.isdir(os.path.join(DATASET_DIR, d))
])
print(f"Train dataset: {DATASET_DIR}")
print(f"Test  dataset: {KAGGLE_TEST_DIR or 'not found (will use split from train)'}")
print(f"Folders found: {all_folders}")

# Count images per letter
image_paths = []
image_labels = []
for letter in LETTERS:
    folder = os.path.join(DATASET_DIR, letter)
    if not os.path.isdir(folder):
        print(f"  WARNING: folder for '{letter}' not found, skipping")
        continue
    files = [f for f in os.listdir(folder) if f.lower().endswith((".jpg", ".jpeg", ".png"))]
    for f in files:
        image_paths.append(os.path.join(folder, f))
        image_labels.append(letter_to_idx[letter])

print(f"\nTotal images: {len(image_paths)}")
print(f"Labels distribution: {dict(Counter(image_labels))}")

## 3. MediaPipe Hand Landmark Extraction

Extract **21 hand landmarks** (x, y) from each image using MediaPipe Hands.
Normalize by centering on the wrist and scaling by hand bounding-box size.

This is much simpler than the TGCN pipeline — no pose, no dual hands, no temporal windows.

In [ ]:
# ============================================================
# Cell 5: MediaPipe Hand Landmark Extraction
# ============================================================
mp_hands = mp.solutions.hands

def extract_hand_landmarks(image_rgb, hands_detector):
    """Extract 21 (x,y) hand landmarks from an RGB image. Returns None if no hand found."""
    result = hands_detector.process(image_rgb)
    if not result.multi_hand_landmarks:
        return None
    # Take the first detected hand
    hand_lm = result.multi_hand_landmarks[0]
    pts = np.array([[lm.x, lm.y] for lm in hand_lm.landmark], dtype=np.float32)
    return pts  # shape: (21, 2)

def normalize_hand(pts):
    """Center on wrist (landmark 0), scale by hand bounding box."""
    wrist = pts[0].copy()
    centered = pts - wrist  # center on wrist

    # Scale by bounding box diagonal
    x_range = centered[:, 0].max() - centered[:, 0].min()
    y_range = centered[:, 1].max() - centered[:, 1].min()
    scale = math.sqrt(x_range**2 + y_range**2)
    if scale < 1e-6:
        return centered
    return centered / scale

print("Extraction functions ready.")

In [ ]:
# ============================================================
# Cell 5b: Restore Landmark Cache from Previous Run (if available)
# ============================================================
import shutil

CACHE_FILE = "/kaggle/working/asl_landmarks_cache.npz"
PREV_CACHE_CANDIDATES = [
    "/kaggle/input/datasets/mitchelkevintu/asl-landmarks-cache/asl_landmarks_cache.npz",
    "/kaggle/input/asl-landmarks-cache/asl_landmarks_cache.npz",
    "/kaggle/input/nimbus-asl-cache/asl_landmarks_cache.npz",
]

restored = False
for candidate in PREV_CACHE_CANDIDATES:
    if os.path.isfile(candidate):
        print(f"Found previous cache: {candidate}")
        shutil.copy2(candidate, CACHE_FILE)
        restored = True
        break

if not restored:
    print("No previous landmark cache found — extraction will run from scratch.")
    print("Tip: after this run, save output as a dataset to skip extraction next time.")

In [ ]:
# ============================================================
# Cell 6: Extract Landmarks from All Images (~5-10 min on Kaggle)
# ============================================================
if os.path.isfile(CACHE_FILE):
    print("Loading cached landmarks...")
    data = np.load(CACHE_FILE)
    X_all = data["X"]
    Y_all = data["Y"]
    print(f"Loaded from cache: {X_all.shape[0]} samples")
else:
    print("Extracting landmarks with MediaPipe...")
    hands_det = mp_hands.Hands(
        static_image_mode=True, max_num_hands=1, min_detection_confidence=0.3
    )

    X_all, Y_all = [], []
    skipped = 0

    for i in tqdm(range(len(image_paths)), desc="Extracting", ncols=100):
        img = cv2.imread(image_paths[i])
        if img is None:
            skipped += 1
            continue
        img_rgb = cv2.cvtColor(img, cv2.COLOR_BGR2RGB)
        pts = extract_hand_landmarks(img_rgb, hands_det)
        if pts is None:
            skipped += 1
            continue
        pts = normalize_hand(pts)
        X_all.append(pts.flatten())  # (42,)
        Y_all.append(image_labels[i])

    hands_det.close()
    X_all = np.array(X_all, dtype=np.float32)
    Y_all = np.array(Y_all, dtype=np.int64)

    # Save cache
    np.savez_compressed(CACHE_FILE, X=X_all, Y=Y_all)
    print(f"\nExtracted: {len(X_all)}, Skipped (no hand detected): {skipped}")
    print(f"Cache saved: {CACHE_FILE}")

print(f"Dataset shape: X={X_all.shape}, Y={Y_all.shape}")
print(f"Labels distribution: {dict(Counter(Y_all.tolist()))}")

## 4. Train/Val/Test Split

In [ ]:
# ============================================================
# Cell 7: Train/Val/Test Split (70/15/15)
# ============================================================
from sklearn.model_selection import train_test_split

X_trainval, X_test, Y_trainval, Y_test = train_test_split(
    X_all, Y_all, test_size=0.15, random_state=42, stratify=Y_all
)
X_train, X_val, Y_train, Y_val = train_test_split(
    X_trainval, Y_trainval, test_size=0.176, random_state=42, stratify=Y_trainval  # 0.176 of 85% ≈ 15%
)

print(f"Train: {X_train.shape}, Val: {X_val.shape}, Test: {X_test.shape}")
print(f"Train labels: {dict(Counter(Y_train.tolist()))}")

In [ ]:
# ============================================================
# Cell 8: Data Augmentation (3x)
# ============================================================
def augment_landmarks(flat_pts):
    """Augment a (42,) flat landmark vector."""
    pts = flat_pts.reshape(NUM_HAND_LANDMARKS, 2).copy()

    # Random scale (0.85 - 1.15)
    if np.random.rand() < 0.6:
        scale = np.random.uniform(0.85, 1.15)
        pts *= scale

    # Random rotation (-15 to +15 degrees)
    if np.random.rand() < 0.5:
        angle = np.random.uniform(-15, 15) * np.pi / 180
        cos_a, sin_a = np.cos(angle), np.sin(angle)
        rotated = pts.copy()
        rotated[:, 0] = pts[:, 0] * cos_a - pts[:, 1] * sin_a
        rotated[:, 1] = pts[:, 0] * sin_a + pts[:, 1] * cos_a
        pts = rotated

    # Random shift
    if np.random.rand() < 0.5:
        shift = np.random.uniform(-0.05, 0.05, size=2).astype(np.float32)
        pts += shift

    # Gaussian noise
    if np.random.rand() < 0.5:
        noise = np.random.normal(0, 0.01, pts.shape).astype(np.float32)
        pts += noise

    return pts.flatten()

NUM_AUG_ROUNDS = 3
print(f"Original training samples: {len(X_train)}")
X_aug, Y_aug = [], []
for r in range(NUM_AUG_ROUNDS):
    for i in range(len(X_train)):
        X_aug.append(augment_landmarks(X_train[i]))
        Y_aug.append(Y_train[i])
    print(f"  Augmentation round {r+1}/{NUM_AUG_ROUNDS} done")

X_train = np.concatenate([X_train, np.array(X_aug, dtype=np.float32)], axis=0)
Y_train = np.concatenate([Y_train, np.array(Y_aug, dtype=np.int64)], axis=0)

perm = np.random.permutation(len(X_train))
X_train = X_train[perm]
Y_train = Y_train[perm]

print(f"Augmented training samples: {len(X_train)}")
del X_aug, Y_aug
gc.collect()

## 5. MLP Model

A simple 3-layer MLP: `42 → 256 → 128 → 26`.
This is all you need for static landmark classification — no GCN, no attention, no temporal modeling.

In [ ]:
# ============================================================
# Cell 9: MLP Model
# ============================================================
class ASLAlphabetMLP(nn.Module):
    def __init__(self, input_dim=42, num_classes=26):
        super().__init__()
        self.net = nn.Sequential(
            nn.Linear(input_dim, 256),
            nn.BatchNorm1d(256),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(256, 128),
            nn.BatchNorm1d(128),
            nn.ReLU(),
            nn.Dropout(0.3),
            nn.Linear(128, num_classes),
        )

    def forward(self, x):
        return self.net(x)

model = ASLAlphabetMLP(input_dim=NUM_FEATURES, num_classes=NUM_CLASSES).to(DEVICE)

total_params = sum(p.numel() for p in model.parameters())
print(f"Model parameters: {total_params:,}")
print(f"Input shape: [batch, {NUM_FEATURES}]")
print(f"Output: {NUM_CLASSES} classes")
print(f"\nEstimated ONNX size: ~{total_params * 4 / 1024:.0f} KB")

In [ ]:
# ============================================================
# Cell 10: Dataset & DataLoaders
# ============================================================
class ASLDataset(Dataset):
    def __init__(self, X, Y):
        self.X = torch.FloatTensor(X)
        self.Y = torch.LongTensor(Y)

    def __len__(self):
        return len(self.Y)

    def __getitem__(self, idx):
        return self.X[idx], self.Y[idx]

BATCH_SIZE = 128

train_ds = ASLDataset(X_train, Y_train)
val_ds = ASLDataset(X_val, Y_val)
test_ds = ASLDataset(X_test, Y_test)

train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, drop_last=True)
val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False)
test_loader = DataLoader(test_ds, batch_size=BATCH_SIZE, shuffle=False)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}, Test batches: {len(test_loader)}")

## 6. Training Loop

In [ ]:
# ============================================================
# Cell 11: Training
# ============================================================
EPOCHS = 100
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-4
PATIENCE = 15

criterion = nn.CrossEntropyLoss(label_smoothing=0.05)
optimizer = optim.AdamW(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
scheduler = optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=EPOCHS, eta_min=1e-6)

best_val_acc = 0.0
best_epoch = 0
patience_counter = 0
best_state = None

history = {"train_loss": [], "train_acc": [], "val_loss": [], "val_acc": []}

for epoch in range(1, EPOCHS + 1):
    # --- Train ---
    model.train()
    train_loss, train_correct, train_total = 0, 0, 0
    for X_batch, Y_batch in train_loader:
        X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
        optimizer.zero_grad()
        logits = model(X_batch)
        loss = criterion(logits, Y_batch)
        loss.backward()
        torch.nn.utils.clip_grad_norm_(model.parameters(), max_norm=1.0)
        optimizer.step()

        train_loss += loss.item() * X_batch.size(0)
        train_correct += (logits.argmax(1) == Y_batch).sum().item()
        train_total += X_batch.size(0)

    scheduler.step()

    # --- Validate ---
    model.eval()
    val_loss, val_correct, val_total = 0, 0, 0
    with torch.no_grad():
        for X_batch, Y_batch in val_loader:
            X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
            logits = model(X_batch)
            loss = criterion(logits, Y_batch)
            val_loss += loss.item() * X_batch.size(0)
            val_correct += (logits.argmax(1) == Y_batch).sum().item()
            val_total += X_batch.size(0)

    t_loss = train_loss / max(train_total, 1)
    t_acc = train_correct / max(train_total, 1)
    v_loss = val_loss / max(val_total, 1)
    v_acc = val_correct / max(val_total, 1)

    history["train_loss"].append(t_loss)
    history["train_acc"].append(t_acc)
    history["val_loss"].append(v_loss)
    history["val_acc"].append(v_acc)

    if v_acc > best_val_acc:
        best_val_acc = v_acc
        best_epoch = epoch
        patience_counter = 0
        best_state = {k: v.cpu().clone() for k, v in model.state_dict().items()}
    else:
        patience_counter += 1

    if epoch % 5 == 0 or epoch == 1 or patience_counter == 0:
        lr = scheduler.get_last_lr()[0]
        print(f"Epoch {epoch:3d}/{EPOCHS} | "
              f"Train Loss: {t_loss:.4f} Acc: {t_acc:.3f} | "
              f"Val Loss: {v_loss:.4f} Acc: {v_acc:.3f} | "
              f"LR: {lr:.6f} | Best: {best_val_acc:.3f} @ {best_epoch}")

    if patience_counter >= PATIENCE:
        print(f"\nEarly stopping at epoch {epoch}. Best val acc: {best_val_acc:.3f} at epoch {best_epoch}")
        break

if best_state is not None:
    model.load_state_dict(best_state)
    model.to(DEVICE)

print(f"\nTraining complete. Best validation accuracy: {best_val_acc:.3f} at epoch {best_epoch}")

In [ ]:
# ============================================================
# Cell 12: Plot Training History
# ============================================================
try:
    import matplotlib.pyplot as plt

    fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(14, 5))

    ax1.plot(history["train_loss"], label="Train")
    ax1.plot(history["val_loss"], label="Val")
    ax1.set_title("Loss")
    ax1.set_xlabel("Epoch")
    ax1.legend()
    ax1.grid(True)

    ax2.plot(history["train_acc"], label="Train")
    ax2.plot(history["val_acc"], label="Val")
    ax2.set_title("Accuracy")
    ax2.set_xlabel("Epoch")
    ax2.legend()
    ax2.grid(True)

    plt.tight_layout()
    plt.show()
except ImportError:
    print("matplotlib not available, skipping plots")

In [ ]:
# ============================================================
# Cell 13: Test Set Evaluation
# ============================================================
model.eval()
test_correct, test_total = 0, 0
all_preds, all_labels = [], []

with torch.no_grad():
    for X_batch, Y_batch in test_loader:
        X_batch, Y_batch = X_batch.to(DEVICE), Y_batch.to(DEVICE)
        logits = model(X_batch)
        preds = logits.argmax(1)
        test_correct += (preds == Y_batch).sum().item()
        test_total += X_batch.size(0)
        all_preds.extend(preds.cpu().numpy())
        all_labels.extend(Y_batch.cpu().numpy())

test_acc = test_correct / max(test_total, 1)
print(f"Test accuracy: {test_acc:.3f} ({test_correct}/{test_total})")

# Per-class accuracy
from collections import defaultdict
class_correct = defaultdict(int)
class_total = defaultdict(int)
for pred, label in zip(all_preds, all_labels):
    class_total[label] += 1
    if pred == label:
        class_correct[label] += 1

print(f"\nPer-class accuracy:")
for idx in sorted(class_total.keys()):
    letter = idx_to_letter[idx]
    acc = class_correct[idx] / class_total[idx] if class_total[idx] > 0 else 0
    print(f"  {letter}: {acc:.3f} ({class_correct[idx]}/{class_total[idx]})")

## 7. Export to ONNX

Produces two files for `web/public/`:
- `asl_alphabet.onnx` → new letter classifier model
- `wlasl_labels.json` → A-Z label mapping (same JSON format as before)

**Input shape**: `[1, 42]` (21 landmarks × 2 coords)
**Output shape**: `[1, 26]` (logits for A-Z)

In [ ]:
# ============================================================
# Cell 14: Export ONNX + Labels
# ============================================================
OUTPUT_DIR = "/kaggle/working/nimbus_model"
os.makedirs(OUTPUT_DIR, exist_ok=True)

model.eval()
model.cpu()

dummy_input = torch.randn(1, NUM_FEATURES)

onnx_path = os.path.join(OUTPUT_DIR, "asl_alphabet.onnx")

export_kwargs = dict(
    export_params=True,
    opset_version=14,
    do_constant_folding=True,
    input_names=["input"],
    output_names=["output"],
    dynamic_axes={"input": {0: "batch"}, "output": {0: "batch"}},
)

try:
    torch.onnx.export(model, dummy_input, onnx_path, dynamo=False, **export_kwargs)
except TypeError:
    torch.onnx.export(model, dummy_input, onnx_path, **export_kwargs)

# Validate ONNX
import onnx
onnx_model = onnx.load(onnx_path)
onnx.checker.check_model(onnx_model)
print(f"ONNX model exported: {onnx_path}")
print(f"  Size: {os.path.getsize(onnx_path) / 1024:.1f} KB")
print(f"  Input:  {[d.dim_value or d.dim_param for d in onnx_model.graph.input[0].type.tensor_type.shape.dim]}")
print(f"  Output: {[d.dim_value or d.dim_param for d in onnx_model.graph.output[0].type.tensor_type.shape.dim]}")

# Validate with onnxruntime
import onnxruntime as ort
sess = ort.InferenceSession(onnx_path)
test_out = sess.run(None, {"input": dummy_input.numpy()})
print(f"  ORT test output shape: {test_out[0].shape}")
assert test_out[0].shape == (1, NUM_CLASSES), f"Unexpected shape {test_out[0].shape}"

# Export labels (same JSON format as before)
labels_path = os.path.join(OUTPUT_DIR, "wlasl_labels.json")
labels_json = {str(i): letter for i, letter in enumerate(LETTERS)}
with open(labels_path, "w") as f:
    json.dump(labels_json, f, indent=2)
print(f"\nLabels exported: {labels_path}")
print(json.dumps(labels_json, indent=2))

In [ ]:
# ============================================================
# Cell 15: Save PyTorch Checkpoint (optional)
# ============================================================
checkpoint_path = os.path.join(OUTPUT_DIR, "asl_alphabet_mlp.pt")
torch.save({
    "model_state_dict": model.state_dict(),
    "letters": LETTERS,
    "num_classes": NUM_CLASSES,
    "num_features": NUM_FEATURES,
    "best_val_acc": best_val_acc,
    "best_epoch": best_epoch,
    "test_acc": test_acc,
}, checkpoint_path)
print(f"Checkpoint saved: {checkpoint_path}")

## 8. Download & Deploy

1. Download `asl_alphabet.onnx` and `wlasl_labels.json` from the output
2. Place both in `web/public/` (replacing the old ONNX model)
3. **Web pipeline changes needed** (see below)

### Browser Pipeline Changes

The old pipeline used a 50-frame temporal buffer with 55 keypoints.
The new pipeline is single-frame, hand-only:

| File | Change |
|---|---|
| `wlaslWorker.js` | Update `MODEL_URL` to `asl_alphabet.onnx`, `INPUT_SHAPE` to `[1, 42]`, `INPUT_SIZE` to `42` |
| `useGlossInference.ts` | Remove 50-frame buffer. Send single-frame hand landmarks (21 pts × 2 = 42 floats) directly |
| `slicer.ts` | Add a `handOnly()` export that returns just 21 normalized hand landmarks |
| `useMediaPipeTracking.ts` | No changes needed — already extracts hand landmarks |

New browser flow:
```
Camera → MediaPipe HandLandmarker → 21 landmarks → normalize (wrist-center, scale) → flatten [42] → ONNX worker → softmax → letter
```

In [ ]:
# ============================================================
# Cell 16: Create download links (Kaggle)
# ============================================================
from IPython.display import FileLink, display

print("Download these files and place in web/public/:\n")
display(FileLink(os.path.join(OUTPUT_DIR, "asl_alphabet.onnx")))
display(FileLink(os.path.join(OUTPUT_DIR, "wlasl_labels.json")))
print("\nOptional (for future fine-tuning):")
display(FileLink(os.path.join(OUTPUT_DIR, "asl_alphabet_mlp.pt")))